# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/Users/aaravwadhwani/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
X = df.drop(columns=["Class"]).values
y = df["Class"].values


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)


In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
model = Sequential([
    Dense(64, activation="relu", input_shape=(num_features,)),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax"),
])


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
history = model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1,
)

Epoch 1/20
13/13 [==============================] - 0s 4ms/step - loss: 0.8639 - accuracy: 0.7374 - val_loss: 0.7025 - val_accuracy: 0.8800
Epoch 2/20
13/13 [==============================] - 0s 810us/step - loss: 0.6090 - accuracy: 0.9192 - val_loss: 0.5213 - val_accuracy: 0.9600
Epoch 3/20
13/13 [==============================] - 0s 774us/step - loss: 0.4314 - accuracy: 0.9596 - val_loss: 0.3713 - val_accuracy: 1.0000
Epoch 4/20
13/13 [==============================] - 0s 772us/step - loss: 0.2968 - accuracy: 0.9899 - val_loss: 0.2643 - val_accuracy: 1.0000
Epoch 5/20
13/13 [==============================] - 0s 768us/step - loss: 0.2046 - accuracy: 0.9899 - val_loss: 0.1945 - val_accuracy: 1.0000
Epoch 6/20
13/13 [==============================] - 0s 745us/step - loss: 0.1466 - accuracy: 0.9899 - val_loss: 0.1559 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1072 - accuracy: 0.9899 - val_loss: 0.1263 - val_accuracy: 0.9600
Epoch 8/20

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print("Test accuracy:", test_acc)

y_pred_probs = model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)
print("Classification report:\n", classification_report(y_true, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

Test accuracy: 1.0
Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion matrix:
 [[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
import os

def file_size_kb(path):
    return os.path.getsize(path) / 1024.0

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
base_tflite_path = "model_base.tflite"
with open(base_tflite_path, "wb") as f:
    f.write(tflite_model)
print(f"Base TFLite model size: {file_size_kb(base_tflite_path):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpq_pcjoln/assets


INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpq_pcjoln/assets


Base TFLite model size: 14.07 KB


2026-05-19 23:16:00.208706: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:16:00.208715: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:16:00.208942: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpq_pcjoln
2026-05-19 23:16:00.209233: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:16:00.209236: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpq_pcjoln
2026-05-19 23:16:00.210040: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-19 23:16:00.210285: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 23:16:00.224049: I tensorflow/cc/saved_model/loader.

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        pass

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
        pass

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        pass

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    # <-- Enter your code here <--#
    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_index = input_details[0]["index"]
    output_index = output_details[0]["index"]
    input_dtype = input_details[0]["dtype"]
    output_dtype = output_details[0]["dtype"]
    input_scale, input_zero_point = input_details[0]["quantization"]
    output_scale, output_zero_point = output_details[0]["quantization"]

    y_pred = []
    for i in range(X_test.shape[0]):
        sample = X_test[i:i + 1].astype(np.float32)
        if input_dtype != np.float32:
            if input_scale > 0:
                sample = np.round(sample / input_scale + input_zero_point)
            sample = sample.astype(input_dtype)

        interpreter.set_tensor(input_index, sample)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_index)
        if output_dtype != np.float32:
            if output_scale > 0:
                output_data = (output_data.astype(np.float32) - output_zero_point) * output_scale
        y_pred.append(int(np.argmax(output_data, axis=1)[0]))

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#
    print(classification_report(y_true, y_pred))
    print(confusion_matrix(y_true, y_pred))

In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
quantize_and_evaluate(model, X_test_scaled, y_test_cat, "int8", "model_int8.tflite")
quantize_and_evaluate(model, X_test_scaled, y_test_cat, "float16", "model_float16.tflite")
quantize_and_evaluate(model, X_test_scaled, y_test_cat, "dynamic", "model_dynamic.tflite")


INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpqcice3tp/assets


INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpqcice3tp/assets
/Users/aaravwadhwani/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



INT8 TFLite model size: 5.74 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]


2026-05-19 23:16:00.478914: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:16:00.478923: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:16:00.479004: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpqcice3tp
2026-05-19 23:16:00.479289: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:16:00.479292: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpqcice3tp
2026-05-19 23:16:00.480066: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 23:16:00.491669: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpqcice3tp
2026-05-

INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmp_hmht77h/assets


INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmp_hmht77h/assets
2026-05-19 23:16:00.669127: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:16:00.669134: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:16:00.669201: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmp_hmht77h
2026-05-19 23:16:00.669478: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:16:00.669480: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmp_hmht77h
2026-05-19 23:16:00.670269: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 23:16:00.681948: I tensorflow/cc/saved_model/loader.cc:217] Running initialization


FLOAT16 TFLite model size: 8.95 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmp7pjdy7a4/assets


INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmp7pjdy7a4/assets



DYNAMIC TFLite model size: 8.17 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]


2026-05-19 23:16:00.833477: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:16:00.833485: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:16:00.833547: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmp7pjdy7a4
2026-05-19 23:16:00.833841: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:16:00.833844: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmp7pjdy7a4
2026-05-19 23:16:00.834657: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 23:16:00.846381: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmp7pjdy7a4
2026-05-

## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
batch_size = 8
epochs = 10
end_step = int(np.ceil(len(X_train_scaled) / batch_size) * epochs)
pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step,
)

In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude
pruned_model = Sequential([
    prune_low_magnitude(
        Dense(64, activation="relu", input_shape=(num_features,)),
        pruning_schedule=pruning_schedule,
    ),
    prune_low_magnitude(Dense(32, activation="relu"), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(num_classes, activation="softmax"), pruning_schedule=pruning_schedule),
])

In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
pruned_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
pruning_callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]
pruned_history = pruned_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=pruning_callbacks,
    verbose=1,
)

Epoch 1/10
13/13 [==============================] - 0s 4ms/step - loss: 0.9118 - accuracy: 0.6566 - val_loss: 0.6961 - val_accuracy: 0.8400
Epoch 2/10
13/13 [==============================] - 0s 960us/step - loss: 0.5994 - accuracy: 0.8889 - val_loss: 0.4472 - val_accuracy: 0.9200
Epoch 3/10
13/13 [==============================] - 0s 937us/step - loss: 0.3972 - accuracy: 0.9697 - val_loss: 0.2872 - val_accuracy: 0.9600
Epoch 4/10
13/13 [==============================] - 0s 849us/step - loss: 0.2614 - accuracy: 0.9697 - val_loss: 0.1949 - val_accuracy: 1.0000
Epoch 5/10
13/13 [==============================] - 0s 861us/step - loss: 0.1828 - accuracy: 0.9798 - val_loss: 0.1398 - val_accuracy: 1.0000
Epoch 6/10
13/13 [==============================] - 0s 893us/step - loss: 0.1339 - accuracy: 0.9899 - val_loss: 0.1072 - val_accuracy: 1.0000
Epoch 7/10
13/13 [==============================] - 0s 851us/step - loss: 0.1025 - accuracy: 0.9899 - val_loss: 0.0860 - val_accuracy: 1.0000
Epoch 8/

In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_pruned = converter.convert()
pruned_tflite_path = "model_pruned.tflite"
with open(pruned_tflite_path, "wb") as f:
    f.write(tflite_pruned)
print(f"Pruned TFLite model size: {file_size_kb(pruned_tflite_path):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpgc7tn4wk/assets


INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpgc7tn4wk/assets


Pruned TFLite model size: 14.14 KB


2026-05-19 23:16:01.677757: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:16:01.677765: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:16:01.677829: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpgc7tn4wk
2026-05-19 23:16:01.678049: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:16:01.678051: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpgc7tn4wk
2026-05-19 23:16:01.678527: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 23:16:01.683237: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpgc7tn4wk
2026-05-

In [20]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
pruned_preds = stripped_model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(pruned_preds, axis=1)
y_true = np.argmax(y_test_cat, axis=1)
print("Classification report:\n", classification_report(y_true, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       0.95      0.95      0.95        21
           2       0.93      0.93      0.93        15

    accuracy                           0.96        54
   macro avg       0.96      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54

Confusion matrix:
 [[18  0  0]
 [ 0 20  1]
 [ 0  1 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
student_model = Sequential([
    Dense(32, activation="relu", input_shape=(num_features,)),
    Dense(16, activation="relu"),
    Dense(num_classes, activation="softmax"),
])

In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
teacher_preds_soft = model.predict(X_train_scaled, verbose=0)

In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)

def distillation_loss(y_true_combined, y_pred):

    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)
    alpha = 0.5
    return alpha * hard_loss + (1 - alpha) * soft_loss
    pass

In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
student_model.compile(optimizer="adam", loss=distillation_loss, metrics=["accuracy"])
student_history = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1,
)

Epoch 1/10
13/13 [==============================] - 0s 4ms/step - loss: 1.0047 - accuracy: 0.4040 - val_loss: 0.9071 - val_accuracy: 0.4800
Epoch 2/10
13/13 [==============================] - 0s 841us/step - loss: 0.8737 - accuracy: 0.5556 - val_loss: 0.8184 - val_accuracy: 0.6400
Epoch 3/10
13/13 [==============================] - 0s 822us/step - loss: 0.7737 - accuracy: 0.6667 - val_loss: 0.7384 - val_accuracy: 0.7600
Epoch 4/10
13/13 [==============================] - 0s 792us/step - loss: 0.6796 - accuracy: 0.8182 - val_loss: 0.6594 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 769us/step - loss: 0.5879 - accuracy: 0.8788 - val_loss: 0.5808 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 776us/step - loss: 0.5016 - accuracy: 0.9293 - val_loss: 0.5067 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 764us/step - loss: 0.4139 - accuracy: 0.9697 - val_loss: 0.4340 - val_accuracy: 0.9200
Epoch 8/

In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd = converter.convert()
kd_tflite_path = "model_kd.tflite"
with open(kd_tflite_path, "wb") as f:
    f.write(tflite_kd)
print(f"KD TFLite model size: {file_size_kb(kd_tflite_path):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpcz3nqd1e/assets


INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpcz3nqd1e/assets


KD TFLite model size: 6.10 KB


2026-05-19 23:16:02.149870: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:16:02.149878: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:16:02.149947: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpcz3nqd1e
2026-05-19 23:16:02.150260: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:16:02.150263: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpcz3nqd1e
2026-05-19 23:16:02.151033: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 23:16:02.162812: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpcz3nqd1e
2026-05-

In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
student_test_preds = student_model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(student_test_preds, axis=1)
y_true = np.argmax(y_test_cat, axis=1)
print("Classification report:\n", classification_report(y_true, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

Classification report:
               precision    recall  f1-score   support

           0       0.89      0.94      0.92        18
           1       0.86      0.86      0.86        21
           2       0.93      0.87      0.90        15

    accuracy                           0.89        54
   macro avg       0.89      0.89      0.89        54
weighted avg       0.89      0.89      0.89        54

Confusion matrix:
 [[17  1  0]
 [ 2 18  1]
 [ 0  2 13]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?
   
Among the single-technique compressions tested in parts (b), (c), and (d), Full Integer (INT8) Quantization yielded the smallest model at 5.74 KB, while perfectly maintaining the baseline accuracy of 1.00. Knowledge Distillation also significantly reduced the size to 6.10 KB but resulted in a slight accuracy drop to 0.89. Pruning alone did not reduce the flat TFLite file size (14.14 KB) without secondary compression.

2. **Propose a strategy** that combines or enhances techniques learned so far.

To achieve further size reduction, the best approach is to layer orthogonal compression techniques. Specifically, applying Full Integer (INT8) Quantization to the Knowledge Distilled Student Model. Distillation provides an architectural reduction (fewer total parameters), while quantization provides a precision reduction (reducing the remaining parameters from 32-bit floats to 8-bit integers).

3.  **Implement** your proposed solution. (Done below)

4. **Evaluate** the resulting model using both:
   - TFLite model size: 3.62 KB
   - Classification performance: Accuracy remained at 0.89, identical to the unquantized student model.

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.

Further size reduction was successfully achieved by combining techniques. By quantizing the distilled student model, the final memory footprint dropped to 3.62 KB, making it roughly 74% smaller than the baseline 14.07 KB model. The change that made the biggest difference was moving from 32-bit floating-point weights to 8-bit integer weights on an already condensed architecture. Because the student model's parameters were highly optimized to mimic the teacher's soft labels, the model was robust enough to handle the precision loss of INT8 quantization without sacrificing any additional classification performance beyond the initial distillation drop.

### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [27]:
# <-- (if needed) Enter your code here <--#
quantize_and_evaluate(student_model, X_test_scaled, y_test_cat, "int8", "model_kd_int8.tflite")

INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpg4fn95mg/assets


INFO:tensorflow:Assets written to: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpg4fn95mg/assets



INT8 TFLite model size: 3.62 KB
              precision    recall  f1-score   support

           0       0.89      0.94      0.92        18
           1       0.86      0.86      0.86        21
           2       0.93      0.87      0.90        15

    accuracy                           0.89        54
   macro avg       0.89      0.89      0.89        54
weighted avg       0.89      0.89      0.89        54

[[17  1  0]
 [ 2 18  1]
 [ 0  2 13]]


/Users/aaravwadhwani/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-19 23:16:02.336041: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 23:16:02.336049: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 23:16:02.336121: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpg4fn95mg
2026-05-19 23:16:02.336404: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 23:16:02.336407: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/86/t62q99rn537c17fwg1c0m5c00000gn/T/tmpg4fn95mg
2026-05-19 23:16:02.337174: I tensorflow/cc/saved_model/loader.cc

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
